# Week 12, Day 6: Forge: Week 12 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w012/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. the Hessian is diag(6, 2), so eta must stay below 2 / 6 =", round(2 / 6, 4))
x = np.linspace(-1, 1, 5); print("2. f'' = 12x^2 - 2 at x = 0 is", 12 * 0 - 2, "so no")
print("3. it is sqrt(16 / 64) =", np.sqrt(16 / 64), "times as large")

In [ ]:
rng = np.random.default_rng(2018)
N, T = 50, 2000
prefs = np.linspace(0, np.pi, N, endpoint=False)
theta = rng.uniform(0, np.pi, T)
labels = (np.cos(2 * (theta - np.pi / 4)) > 0).astype(float)
counts = rng.poisson((10 + 20 * np.exp(2 * (np.cos(2 * (theta[:, None] - prefs[None])) - 1))) * 0.2)
train, test = np.arange(1500), np.arange(1500, T)
def sigmoid(u): return 1 / (1 + np.exp(-u))
def xent(w, Z, y):
    p = np.clip(sigmoid(Z @ w), 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
def grad(w, Z, y): return Z.T @ (sigmoid(Z @ w) - y) / len(y)
Z_raw = np.column_stack([counts, np.ones(T)])                    # raw spike counts plus a bias input
print("mean count", counts.mean().round(2), "  largest count", counts.max())

### Exercise: Part A: scaling, conditioning and speed

Build `Z_z`, the counts z-scored with the training trials' mean and standard deviation, plus a bias column. For each design, compute the condition number of $Z^\top Z/(4T)$ on the training trials (the Hessian of the loss at w = 0), set the learning rate to 1.9 divided by its largest eigenvalue, run 200 steps of gradient descent from zero, and record the final training loss (`loss_raw`, `loss_z`) and condition numbers (`cond_raw`, `cond_z`).

In [ ]:
mu, sd = counts[train].mean(axis=0), counts[train].std(axis=0) + 1e-9
Z_z = ...

def run_gd(Z, steps=200):
    Zt, yt = Z[train], labels[train]
    ev = np.linalg.eigvalsh(Zt.T @ Zt / (4 * len(yt)))
    lr = 1.9 / ev.max()
    w = np.zeros(Z.shape[1]); hist = []
    for _ in range(steps):
        w = ...
        hist.append(xent(w, Zt, yt))
    return w, hist, ev.max() / ev.min()

w_raw, h_raw, cond_raw = run_gd(Z_raw)
w_z, h_z, cond_z = run_gd(Z_z)
loss_raw, loss_z = h_raw[-1], h_z[-1]

plt.figure(figsize=(5, 3)); plt.plot(h_raw, label=f"raw counts (condition number {cond_raw:.0f})"); plt.plot(h_z, label=f"z-scored ({cond_z:.0f})")
plt.xlabel("step"); plt.ylabel("training loss"); plt.legend(frameon=False); plt.show()
print(f"after 200 steps: raw {loss_raw:.4f}, z-scored {loss_z:.4f}")

In [ ]:
import numpy as np
def test_cond():
    "z-scoring reduces the condition number by more than a factor of 1,000"
    assert cond_raw > 1000 * cond_z

def test_faster():
    "with each design's own best step, the z-scored decoder reaches a lower training loss in 200 steps"
    assert loss_z < loss_raw - 0.01

test_cond()
test_faster()
print("All checks passed")

### Exercise: Part B: one pass through the data

With the z-scored design and learning rate 1, compare one full-batch gradient step (`loss_gd`) with one epoch of minibatch SGD with batches of 20 in a random order (`loss_sgd`). Both see every training example once; record how many examples each processed (`n_examples_gd`, `n_examples_sgd`). Report test accuracy too.

In [ ]:
Zt, yt = Z_z[train], labels[train]
w = np.zeros(N + 1)
w = w - 1.0 * grad(w, Zt, yt)
loss_gd, n_examples_gd = xent(w, Zt, yt), len(yt)
acc_gd = np.mean((sigmoid(Z_z[test] @ w) > 0.5) == labels[test])

w = np.zeros(N + 1); n_examples_sgd = 0
order = np.random.default_rng(0).permutation(len(yt))
for start in range(0, len(yt), 20):
    b = order[start:start + 20]
    ...
loss_sgd = xent(w, Zt, yt)
acc_sgd = np.mean((sigmoid(Z_z[test] @ w) > 0.5) == labels[test])
print(f"one full-batch step: loss {loss_gd:.4f}, test accuracy {acc_gd:.3f};  one epoch of SGD: loss {loss_sgd:.4f}, test accuracy {acc_sgd:.3f}")

In [ ]:
import numpy as np
def test_budget():
    "both runs use exactly one pass through the training trials"
    assert n_examples_gd == 1500 and n_examples_sgd == 1500

def test_sgd_wins():
    "after one epoch, minibatch SGD has a clearly lower training loss than one full-batch step"
    assert loss_sgd < loss_gd - 0.02

test_budget()
test_sgd_wins()
print("All checks passed")

In [ ]:
X2 = rng.normal(size=(400, 2))
y2 = ((X2[:, 0] > 0) ^ (X2[:, 1] > 0)).astype(float)

def train_net(seed, H=3, steps=3000, lr=0.5):
    r = np.random.default_rng(seed)
    W1, b1, w2, b2 = r.normal(size=(2, H)), np.zeros(H), r.normal(size=H) * 0.5, 0.0
    for _ in range(steps):
        h = np.tanh(X2 @ W1 + b1); p = sigmoid(h @ w2 + b2); d = (p - y2) / len(y2)
        dh = np.outer(d, w2) * (1 - h ** 2)
        W1 -= lr * X2.T @ dh; b1 -= lr * dh.sum(axis=0); w2 -= lr * h.T @ d; b2 -= lr * d.sum()
    p = sigmoid(np.tanh(X2 @ W1 + b1) @ w2 + b2)
    return np.mean((p > 0.5) == y2)
print("one network's accuracy:", train_net(0))

### Exercise: Part C: different starts, different answers

Fit logistic regression to the XOR task (2000 gradient steps with learning rate 0.5 on the two inputs plus a bias) and store its accuracy in `acc_logistic`. Then train 20 networks with seeds 0 to 19 using `train_net` and store their accuracies in `accs`. Plot the spread.

In [ ]:
Zx = np.column_stack([X2, np.ones(len(y2))])
wl = np.zeros(3)
for _ in range(2000):
    wl = ...
acc_logistic = np.mean((sigmoid(Zx @ wl) > 0.5) == y2)
accs = ...

plt.figure(figsize=(5, 3)); plt.plot(sorted(accs), "o"); plt.axhline(acc_logistic, ls="--", color="0.5", label="logistic regression")
plt.xlabel("network (sorted)"); plt.ylabel("training accuracy"); plt.legend(frameon=False); plt.show()
print("logistic regression:", round(acc_logistic, 3), "  networks: from", min(accs), "to", max(accs))

In [ ]:
import numpy as np
def test_logistic():
    "logistic regression cannot solve the XOR task (accuracy below 0.65)"
    assert acc_logistic < 0.65

def test_spread():
    "20 networks from different random starts reach clearly different accuracies"
    assert len(accs) == 20 and max(accs) - min(accs) > 0.1 and max(accs) > 0.88

test_logistic()
test_spread()
print("All checks passed")